# Le bus de communication multi-agents en C#

**Objectifs :** ordonner, filtrer, router et corréler des messages ; distinguer le contrat local des transports concrets. **Prérequis :** collections et classes C#. **Durée :** 45 minutes.

Ce port BCL reprend le contrat de `communication_channels.py`. Il ne lance aucun réseau ni thread. Les identifiants générés sont des UUIDv4 tronqués : leur préfixe indique le type, mais deux messages identiques reçoivent normalement des identifiants différents.

**Série Argumentation** | Précédent : [07-Orchestration](./Argumentation-07-Orchestration-Python.ipynb) | Suivant : [07c-Modes](./Argumentation-07c-Orchestration-Modes-CSharp.ipynb) | [↑ Série Argumentation](README.md)

Le contrat C# est construit à partir du module Python local, lui-même extrait des trois modules du tronc. La source de verite du tronc est :

- `test_channel_interface.py` (10 cas) ;
- `test_message.py` (15 cas) ;
- `test_communication_integration.py` (4 cas).

Soit **29 tests** au total — **pas 12 bancs / 45 cas** comme la version
precedente l'annoncait. Ce carnet rejoue les cas deterministes et documente
les autres en docstring du port vendored.

Trois `ChannelType` n'ont pas d'implementation `Channel` concrete ; leurs
routes ont ete **retirees du routage** (#1571) plutot que cablees a des classes
vides — `EVENT`, `CONTROL`, `SUBSCRIPTION` ne routent plus vers `FEEDBACK` /
`SYSTEM`, ils tombent sur le bus par defaut. Le carnet documente ce
comportement dans les cellules de routage.


In [1]:
// Contrat de messages et canal mémoire ; aucune opération réseau.
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text.Json;

enum MessageType
{
    command,
    information,
    request,
    response,
    @event,
    control,
    publication,
    subscription
}

enum Priority
{
    low,
    normal,
    high,
    critical
}

enum AgentLevel
{
    strategic,
    tactical,
    operational,
    system
}

enum ChannelType
{
    hierarchical,
    collaboration,
    data,
    negotiation,
    feedback,
    system,
    local
}

class Message : IComparable<Message>
{
    public string Id;
    public MessageType Type;
    public string Sender;
    public AgentLevel SenderLevel;
    public Dictionary<string, object> Content;
    public string Recipient, Channel;
    public Priority Priority;
    public Dictionary<string, object> Metadata;
    public DateTime Timestamp;
    public Message(MessageType type, string sender, AgentLevel level, Dictionary<string, object> content, string recipient = null, string channel = null, Priority priority = Priority.normal, Dictionary<string, object> metadata = null, string id = null, DateTime? timestamp = null)
    {
        Type = type;
        Sender = sender;
        SenderLevel = level;
        Content = content;
        Recipient = recipient;
        Channel = channel;
        Priority = priority;
        Metadata = metadata ?? new();
        Id = string.IsNullOrEmpty(id) ? $"{type}-{Guid.NewGuid().ToString("N")[..8]}" : id;
        Timestamp = timestamp ?? DateTime.Now;
    }

    public override bool Equals(object obj) => obj is Message other && Priority == other.Priority && Timestamp == other.Timestamp && Id == other.Id;
    public override int GetHashCode() => HashCode.Combine(Priority, Timestamp, Id);
    // Les priorités fortes sont classées avant les faibles à timestamp quelconque.
    public int CompareTo(Message other)
    {
        int p = other.Priority.CompareTo(Priority);
        return p != 0 ? p : Timestamp.CompareTo(other.Timestamp);
    }

    // La représentation sérialisée garde les noms du contrat de messages.
    public Dictionary<string, object> ToDictionary() => new()
    {
        ["id"] = Id,
        ["type"] = Type.ToString(),
        ["sender"] = Sender,
        ["sender_level"] = SenderLevel.ToString(),
        ["recipient"] = Recipient,
        ["channel"] = Channel,
        ["priority"] = Priority.ToString(),
        ["content"] = Content,
        ["metadata"] = Metadata,
        ["timestamp"] = Timestamp.ToString("O")
    };
    public static Message FromDictionary(Dictionary<string, object> d) => new(Enum.Parse<MessageType>((string)d["type"]), (string)d["sender"], Enum.Parse<AgentLevel>((string)d["sender_level"]), (Dictionary<string, object>)d["content"], (string)d.GetValueOrDefault("recipient"), (string)d.GetValueOrDefault("channel"), Enum.Parse<Priority>((string)d["priority"]), (Dictionary<string, object>)d.GetValueOrDefault("metadata"), (string)d["id"], DateTime.Parse((string)d["timestamp"], null, System.Globalization.DateTimeStyles.RoundtripKind));
    public static Message Response(Message request, AgentLevel level, Dictionary<string, object> content) => new(MessageType.response, string.IsNullOrEmpty(request.Recipient) ? "unknown" : request.Recipient, level, content, request.Sender, request.Channel, request.Priority, new() { ["reply_to"] = request.Id });
}

class LocalChannel
{
    readonly List<Message> queue = new();
    readonly Dictionary<string, (Action<Message> callback, Dictionary<string, object> filter)> subs = new();
    public string Id;
    public Dictionary<string, object> Config;
    public LocalChannel(string id, Dictionary<string, object> config = null)
    {
        Id = id;
        Config = config ?? new();
    }

    public Dictionary<string, object> Info() => new()
    {
        ["id"] = Id,
        ["type"] = "local",
        ["subscribers"] = subs.Count,
        ["pending"] = queue.Count
    };
    static bool MatchValue(string actual, object wanted) => wanted is IEnumerable<string> list && wanted is not string ? list.Contains(actual) : Equals(actual, wanted);
    // Les critères se composent par conjonction, et chaque clé rencontrée doit être reconnue.
    public bool Matches(Message m, Dictionary<string, object> filter)
    {
        foreach (var(key, v)in filter ?? new())
        {
            switch (key)
            {
                case "message_type":
                    if (!MatchValue(m.Type.ToString(), v))
                        return false;
                    break;
                case "sender":
                    if (!MatchValue(m.Sender, v))
                        return false;
                    break;
                case "sender_level":
                    if (!MatchValue(m.SenderLevel.ToString(), v))
                        return false;
                    break;
                case "priority":
                    if (!MatchValue(m.Priority.ToString(), v))
                        return false;
                    break;
                case "content":
                    foreach (var(k, val)in (Dictionary<string, object>)v)
                        if (!m.Content.TryGetValue(k, out var actual) || !Equals(actual, val))
                            return false;
                    break;
                default:
                    throw new ArgumentException($"Clé inconnue : {key}");
            }
        }

        return true;
    }

    public string Subscribe(string id, Action<Message> callback = null, Dictionary<string, object> filter = null)
    {
        subs[id] = (callback, filter ?? new());
        return id;
    }

    public bool Unsubscribe(string id) => subs.Remove(id);
    public bool Send(Message message)
    {
        queue.Add(message);
        // La copie des abonnements laisse chaque callback libre de se désabonner.
        foreach (var subscription in subs.Values.ToArray())
            if (Matches(message, subscription.filter))
                subscription.callback?.Invoke(message);
        return true;
    }

    public List<Message> Pending(string recipient, int? max = null)
    {
        var pending = queue.Where(message => message.Recipient == null || message.Recipient == recipient).ToList();
        if (max is null)
            return pending;
        // Une borne négative omet les derniers éléments comme un slice Python.
        int cap = max.Value < 0 ? pending.Count + max.Value : max.Value;
        return pending.Take(Math.Max(0, cap)).ToList();
    }

    public Message Receive(string recipient)
    {
        // Retirer cet emplacement, même si un autre message lui est égal par identifiant.
        int index = queue.FindIndex(message => message.Recipient == null || message.Recipient == recipient);
        if (index < 0)
            return null;
        var result = queue[index];
        queue.RemoveAt(index);
        return result;
    }

    public int Count => queue.Count;
    public int Subscribers => subs.Count;
}

// Un canal explicite valide précède les règles typées et le canal par défaut.
static ChannelType Route(Message m)
{
    if (Enum.TryParse<ChannelType>(m.Channel, out var explicitChannel) && Enum.IsDefined(explicitChannel) && explicitChannel.ToString() == m.Channel)
        return explicitChannel;
    if (m.Type == MessageType.publication)
        return ChannelType.data;
    if (m.Type == MessageType.information && m.Content.TryGetValue("info_type", out var i) && i is string si && si.Contains("analysis_result"))
        return ChannelType.data;
    if (m.Type == MessageType.request && m.Content.TryGetValue("request_type", out var r) && r is string sr && sr.Contains("assistance"))
        return ChannelType.collaboration;
    return ChannelType.hierarchical;
}

Console.WriteLine($"Types : {string.Join(", ", Enum.GetNames<MessageType>())} ; priorités : {string.Join(", ", Enum.GetNames<Priority>())}");
Console.WriteLine($"Niveaux : {string.Join(", ", Enum.GetNames<AgentLevel>())} ; canaux : {string.Join(", ", Enum.GetNames<ChannelType>())}");

Types : command, information, request, response, event, control, publication, subscription ; priorités : low, normal, high, critical


Niveaux : strategic, tactical, operational, system ; canaux : hierarchical, collaboration, data, negotiation, feedback, system, local


**Lecture du resultat.** Huit types de message, quatre priorites, quatre
niveaux d'agent, sept types de canal — mais **trois types de canal n'ont
aucune implementation** (`negotiation`, `feedback`, `system`). Le tronc les
declare dans l'enumere mais les a **retires du routage** (#1571) plutot que
de les cabler a des classes vides : un canal promis sans implementation est
un mensonge du bus — les messages y arriveraient sans jamais etre delivres.
C'est une lecon d'architecture : enumerer un type ne cree pas un transport.

## 1. Le format message — priorite inversee, identité générée

Un `Message` porte type / emetteur / niveau / priorite / contenu, un identifiant préfixé par le type et complété par un UUIDv4 (`command-<hex8>`), et un **ordonnancement total inverse** :
`CompareTo` est ecrit pour que la priorite la plus HAUTE soit la plus « petite »
— `OrderBy(x => x)` sort donc CRITICAL en premier, meme arrive en dernier. A
priorite egale, le plus ancien passe d'abord (FIFO).

In [2]:
// Priorité décroissante, puis ordre FIFO en cas d’égalité.
var t0 = new DateTime(2026, 1, 1, 12, 0, 0);
Message Make(Priority p, int seconds) => new(MessageType.information, "s", AgentLevel.system, new(), priority: p, timestamp: t0.AddSeconds(seconds));
var demo = new Message(MessageType.command, "strategic-1", AgentLevel.strategic, new() { ["task"] = "analyse" });
Console.WriteLine($"Identifiant COMMAND : {demo.Id}");
var ms = new[]
{
    Make(Priority.low, 0),
    Make(Priority.critical, 5),
    Make(Priority.high, 3),
    Make(Priority.normal, 2)
};
foreach (var m in ms.OrderBy(x => x))
    Console.WriteLine($"{m.Priority} {m.Timestamp:HH:mm:ss}");
var a = Make(Priority.normal, 0);
var b = Make(Priority.normal, 5);
Console.WriteLine($"FIFO : {new[] { b, a }.OrderBy(x => x).First() == a}");
var same1 = Make(Priority.normal, 0);
var same2 = Make(Priority.normal, 0);
Console.WriteLine($"Messages identiques, identifiants distincts : {same1.Id != same2.Id}");
var copy = Message.FromDictionary(a.ToDictionary());
Console.WriteLine($"Aller-retour : {copy.Id == a.Id && copy.Priority == a.Priority && copy.Timestamp == a.Timestamp}");
// Trois propriétés sont optionnelles à la lecture d'un message sérialisé.
var minimal = a.ToDictionary();
minimal.Remove("recipient");
minimal.Remove("channel");
minimal.Remove("metadata");
var reloaded = Message.FromDictionary(minimal);
Console.WriteLine($"Lecture sans champs facultatifs : {reloaded.Recipient == null && reloaded.Channel == null && reloaded.Metadata.Count == 0}");

Identifiant COMMAND : command-65aa24ee


critical 12:00:05


high 12:00:03


normal 12:00:02


low 12:00:00


FIFO : True


Messages identiques, identifiants distincts : True


Aller-retour : True


Lecture sans champs facultatifs : True


**Lecture du resultat.** L'id commence par `command-` : le prefixe dit deja
la nature du message. L'ordonnancement est **inverse** : CRITICAL (arrive
dernier, a 12:00:05) sort en premier, LOW (arrive premier) sort en dernier —
`CompareTo` compare les priorites en les inversant, puis les timestamps en FIFO
a priorite egale. C'est un choix deliberement contre-intuitif : la priorite
la plus haute est la plus « petite » pour que `OrderBy(x => x)` la place en tete.
La convention est simple et verifiable — mais elle impose une discipline :
toute opération d’ordre doit respecter `CompareTo` plutôt qu’une comparaison manuelle qui ignorerait cette convention.

Le banc `dict_round_trip` complete le tableau : `ToDictionary()` puis
`FromDictionary()` preserve l'identite (l'egalite porte sur id, priorite et
horodatage). Cette conversion en dictionnaire conserve les champs ; un transport entre processus doit en plus choisir un format de sérialisation adapté.

## 2. Le contrat de canal — filtres fail-loud

Un canal est identifie par un id et un type ; il maintient une file d'attente
et un registre d'abonnes avec filtres. Le matcher est **fail-loud** : une cle
hors contrat leve `ArgumentException` plutot que d'etre ignoree — l'ignorance
silencieuse elargit le filtre sans que personne ne le voie (incident #2161 :
trois defauts coexistaient sans un seul test rouge).

In [3]:
// Les critères testent séparément le type, la priorité et les clés du contenu.
var ch = new LocalChannel("test");
var m = new Message(MessageType.command, "tactical-1", AgentLevel.tactical, new() { ["task"] = "analyse" }, priority: Priority.high, timestamp: t0);
foreach (var f in new[]
{
    new Dictionary<string, object>
    {
        {
            "message_type",
            "command"
        }
    },
    new()
    {
        {
            "message_type",
            "information"
        }
    },
    new()
    {
        {
            "message_type",
            new[]
            {
                "command",
                "information"
            }
        }
    },
    new()
    {
        {
            "priority",
            "high"
        }
    },
    new()
    {
        {
            "content",
            new Dictionary<string, object>
            {
                {
                    "task",
                    "analyse"
                }
            }
        }
    },
    new()
    {
        {
            "content",
            new Dictionary<string, object>
            {
                {
                    "task",
                    "autre"
                }
            }
        }
    }
}

)
    Console.WriteLine(ch.Matches(m, f));
try
{
    ch.Matches(m, new() { { "bogus_key", "value" } });
}
catch (ArgumentException e)
{
    Console.WriteLine(e.Message);
}

True


False


True


True


True


False


Clé inconnue : bogus_key


**Lecture du resultat.** Le contrat parle en **chaines** : `command`, pas
l'enumere — une liste est un OU logique, un scalaire un ET strict. Le filtre
de contenu est une sous-correspondance : chaque cle du filtre doit exister
dans le contenu du message avec la meme valeur. Et le comportement critique :
une cle inconnue leve `ArgumentException` au lieu d'etre ignoree. Sans ce fail-loud,
un filtre `{"priorite": "high"}` (faute de frappe) passerait tous les
messages — un filtre mort qui a l'air vivant. Le cout d'une erreur levee est
visible ; le cout d'un filtre elargi est invisible jusqu'au incident.

Le banc `topic_filter_cases` du sas complete le contrat : les filtres de
topic (pour le canal pub/sub) suivent la meme logique — chaines, listes,
sous-correspondance de contenu. La famille de filtres est coherente entre
tous les canaux : un abonne qui sait filtrer sur le canal local sait filtrer
sur le canal pub/sub, sans apprendre une deuxieme syntaxe.

## 3. Le routage — le middleware decide, sans routes mortes

`determine_channel` est la seule partie du middleware portee : les canaux
concrets et le protocole requete-reponse restent sur le tronc. Les regles
sont simples : canal explicite > type de message > contenu > bus par defaut.

### Exercice 1 — Filtrer les commandes critiques

Composez un filtre pour les commandes `CRITICAL` d'un agent `STRATEGIC`. Le cas positif doit passer, et l'autre message doit être bloqué.

In [4]:
// Exercice 1 : composer type command, priorité critical et niveau strategic.
var filtre = new Dictionary<string, object>(); // TODO étudiant
var mOk = new Message(MessageType.command, "chef", AgentLevel.strategic, new(), priority: Priority.critical);
var mKo = new Message(MessageType.information, "agent", AgentLevel.operational, new(), priority: Priority.low);
Console.WriteLine(filtre.Count == 0 ? "Exercice à compléter" : $"OK passe : {ch.Matches(mOk, filtre)} ; KO bloqué : {!ch.Matches(mKo, filtre)}");

Exercice à compléter


In [5]:
ChannelType Probe(MessageType t, Dictionary<string, object> c = null, string channel = null) => Route(new(t, "s", AgentLevel.system, c ?? new(), channel: channel));
foreach (var t in Enum.GetValues<MessageType>())
    Console.WriteLine($"{t} -> {Probe(t)}");
Console.WriteLine($"Résultat -> {Probe(MessageType.information, new() { { "info_type", "analysis_result" } })} ; assistance -> {Probe(MessageType.request, new() { { "request_type", "assistance" } })}");
Console.WriteLine($"Canal explicite -> {Probe(MessageType.command, channel: "data")} ; invalide -> {Probe(MessageType.command, channel: "bogus")}");

command -> hierarchical


information -> hierarchical


request -> hierarchical


response -> hierarchical


event -> hierarchical


control -> hierarchical


publication -> data


subscription -> hierarchical


Résultat -> data ; assistance -> collaboration


Canal explicite -> data ; invalide -> hierarchical


**Lecture du resultat.** Les commandes et reponses vont sur le canal
hierarchique ; les informations portant un resultat d'analyse vont sur le
canal donnees ; les requetes d'assistance vont sur le canal collaboration ;
les publications vont sur le canal donnees. Et le comportement #1571 :
`EVENT` (comme `CONTROL` et `SUBSCRIPTION`) ne route **plus** vers
`feedback` ou `system` — ces canaux n'ont pas d'implementation, les messages
y arriveraient sans jamais etre delivres. Ils tombent sur le bus par defaut
(hierarchique). Un canal explicite prime toujours ; un canal invalide
retombe sur les regles. C'est une table de routage qui ne promet que des
transports qui existent.

Le banc `routing_cases` du sas compte 10 cas — les cas rejoués ici et
des cas de repli (canal explicite prime, canal invalide retombe, type
inconnu tombe sur le bus par defaut). Le middleware du tronc ajoute la
statistique (`stats["messages_sent"]`, `by_channel`) et le registre de
gestionnaires par type — le port ne les porte pas : ce sont des compteurs
d'observabilite, pas des garanties du contrat. Le carnet source les cite ;
ce carnet les mesure par l'usage.

## 4. Le canal local — file d'attente, abonnes, notification synchrone

`LocalChannel` est l'implementation minimale du contrat : file d'attente en
memoire, notification synchrone des abonnes au moment de l'envoi. Il sert de
reference pour les quatre canaux concrets du tronc, qui ajoutent chacun leur
semantique (persistance, topics, hierarchie).

### Exercice 2 : router EVENT metric vers data sans changer le routeur de base.

Complétez ce squelette puis comparez votre résultat au contrat.

In [6]:
// Exercice 2 : router EVENT metric vers data sans changer le routeur de base.
ChannelType? RouteEvent(Message x)
{
    if (x.Type == MessageType.@event && x.Content.TryGetValue("event_type", out var value) && object.Equals(value, "metric"))
        return null; // TODO étudiant
    return Route(x);
}

Console.WriteLine(RouteEvent(new(MessageType.@event, "s", AgentLevel.system, new() { ["event_type"] = "metric" }))?.ToString() ?? "Exercice à compléter");

Exercice à compléter


In [7]:
// Deux abonnements mesurent séparément le journal et les alertes.
var bus = new LocalChannel("bus");
var received = new List<Message>();
var urgent = new List<Message>();
bus.Subscribe("logger", received.Add);
bus.Subscribe("urgent", urgent.Add, new() { ["priority"] = "critical" });
bus.Send(new(MessageType.information, "agent-1", AgentLevel.operational, new() { ["data"] = "result" }, recipient: "agent-2"));
bus.Send(new(MessageType.command, "chef", AgentLevel.strategic, new() { ["task"] = "stop" }, priority: Priority.critical));
Console.WriteLine($"Journal : {received.Count} | urgentes : {urgent.Count} | notifications totales : {received.Count + urgent.Count}");
Console.WriteLine($"En attente avant : {bus.Pending("agent-2").Count} ; reçu : {bus.Receive("agent-2").Id} ; après : {bus.Pending("agent-2").Count}");
Console.WriteLine("Info canal : " + string.Join(", ", bus.Info().Select(x => $"{x.Key}={x.Value}")));

Journal : 2 | urgentes : 1 | notifications totales : 3


En attente avant : 2 ; reçu : information-9085e8a4 ; après : 1


Info canal : id=bus, type=local, subscribers=2, pending=1


**Lecture du resultat.** Le `logger` recoit les deux messages (pas de filtre),
l'abonne `urgent` n'en recoit qu'un (filtre `critical`). Les messages restent
en file tant qu'ils ne sont pas lus — `Receive` les consomme. C'est
un bus synchrone : l'envoi notifie immediatement, la reception consomme. Les
quatre canaux concrets du tronc ajoutent chacun une semantique au-dessus de
ce contrat : le canal hierarchique ajoute des niveaux d'autorisation, le canal
collaboration ajoute des conversations, le canal donnees ajoute la
persistance, le canal pub/sub ajoute des topics. Tous partagent le meme
contrat — c'est ce qui les rend interchangeables.

Le banc `collaboration_cases` du sas mesure ce que le canal collaboration
ajoute : une conversation (un fil conducteur entre plusieurs messages), des
participants, et un etat partage qui survit a l'envoi. Le banc `data_cases`
mesure la persistance : un message envoye sur le canal donnees survit a la
reception — il est stocke, pas consomme. Le banc `pubsub_cases` mesure les
topics : un abonne ne recoit que les messages dont le topic correspond a
son abonnement. Ces trois semantiques sont des extensions du meme contrat —
le canal local n'en a besoin d'aucune pour etre utilisable.

## 5. La correlation requete-reponse — sans threads, sans timeouts

Le protocole requete-reponse complet du tronc (`RequestResponseProtocol`)
introduit des threads et des timeouts — hors d'un port deterministe. Ce qui
est porte est la **convention de correlation** : une reponse inverse emetteur
et destinataire, et porte `reply_to` l'id de la requete.

In [8]:
var req = new Message(MessageType.request, "strategic-1", AgentLevel.strategic, new() { ["request_type"] = "assistance" }, recipient: "tactical-1", channel: "collaboration", priority: Priority.high, id: "request-demo", timestamp: t0);
var resp = Message.Response(req, AgentLevel.tactical, new() { ["result"] = "ok" });
Console.WriteLine($"Correlation : {resp.Metadata["reply_to"]} ; inversion : {resp.Sender == req.Recipient && resp.Recipient == req.Sender}");
Console.WriteLine($"Canal et priorité hérités : {resp.Channel == req.Channel && resp.Priority == req.Priority}");
var noRecipient = Message.Response(new(MessageType.request, "demandeur", AgentLevel.system, new(), recipient: ""), AgentLevel.system, new());
Console.WriteLine($"Sans destinataire explicite, émetteur de la réponse : {noRecipient.Sender}.");

Correlation : request-demo ; inversion : True


Canal et priorité hérités : True


Sans destinataire explicite, émetteur de la réponse : unknown.


**Lecture du resultat.** La reponse porte `reply_to: request-demo` — le
lien entre requete et reponse est un champ de metadata, pas un thread ni une
socket. L'emetteur de la reponse est le destinataire de la requete, et
inversement. C'est une convention de correlation minimale : elle suffit a
reconstruire une conversation sans etat partage. Le protocole complet du
tronc ajoute les timeouts et les reessais — mais la correlation, elle, est
deja la.

Le banc `request_response_cases` du sas compte 3 cas : la correlation par
`reply_to`, le timeout (une requete sans reponse leve `RequestTimeoutError`),
et le reessai (`retry_count`). Les deux derniers sont hors d'un port
deterministe — ils introduisent le temps reel et les threads. Ce qui est
porte est ce qui ne depend pas du temps : la convention de correlation.
Le timeout est un mecanisme de robustesse, pas une garantie de contrat.

### Exercice 3 : envoyer la requête, la recevoir, répondre et vérifier la corrélation.

Complétez ce squelette puis comparez votre résultat au contrat.

In [9]:
// Exercice 3 : envoyer la requête, la recevoir, répondre et vérifier la corrélation.
var exoBus = new LocalChannel("exo3");
var exoReq = new Message(MessageType.request, "agent-A", AgentLevel.tactical, new() { ["query"] = "statut" }, recipient: "agent-B", timestamp: t0);
// TODO étudiant : compléter la conversation.
Console.WriteLine("Exercice à compléter");

Exercice à compléter


## A retenir

1. **Le format message est un contrat, pas une structure de donnees.** L'identifiant
   porte un préfixe de type et un suffixe UUIDv4, la priorite est inversee pour `OrderBy(x => x)`, la correlation
   est un champ de metadata — chaque choix est une convention verifiable,
   pas un accident d'implementation.
2. **Un filtre qui ignore une cle est un filtre elargi.** Le fail-loud
   (`ArgumentException` sur cle inconnue) est la difference entre un filtre qui
   dit ce qu'il fait et un filtre qui fait ce qu'on croit qu'il dit.
3. **Le routage ne promet que des transports qui existent.** Trois types de
   canal sans implementation ont ete retires du routage (#1571) plutot que
   cables a des classes vides — un canal promis sans implementation est un
   mensonge du bus.
4. **Le contrat rend les canaux interchangeables.** Hierarchique,
   collaboration, donnees, pub/sub : tous implementent le meme contrat
   (`send_message` / `Receive` / `subscribe` / `get_pending_messages`),
   tous sont testables avec `LocalChannel`.

**Position dans la serie** : `Orchestration_Modes` compare les architectures
d'agents par le budget ; ce carnet mesure ce qui les relie — le bus de
communication. La suite de la distillation sas est `knowledge_base` (la
memoire du debat) et `dialogue_protocols` (les regles de l'echange).

Le bus est le tissu conjonctif : sans lui, les agents ne se parlent pas.
La memoire (`knowledge_base`) enregistre ce qui a ete dit ; les protocoles
(`dialogue_protocols`) decident qui peut dire quoi ; les cadres
(`Dung_AF_Semantics`) arbitrent ce qui survit. Quatre niveaux d'un meme
debat — le bus est le premier.

***

> Carnet issu de l'EPIC #4960.


## Vérifiez votre compréhension

1. Pourquoi deux messages identiques peuvent-ils avoir des identifiants différents ?
2. Que fait le filtre lorsqu'une clé lui est inconnue ?
3. Pourquoi la priorité critique apparaît-elle en premier ?
4. Sur quoi repose la corrélation requête-réponse ?

<details><summary>Éléments de réponse</summary>

1. Le suffixe est tiré d'un UUIDv4 ; l'appelant peut aussi fournir l'identifiant.
2. Il lève `ArgumentException`, sans élargir silencieusement le filtre.
3. `CompareTo` inverse la comparaison des priorités, puis compare les dates pour le FIFO.
4. La réponse inverse les acteurs et porte l'identifiant de la requête dans `reply_to`.
</details>

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place de ce carnet dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet.

- Le rung [7-Orchestration](./Argumentation-07-Orchestration-Python.ipynb) pour les agents que ce bus relie.
- Le rung [4b-Knowledge-Base](./Argumentation-04b-Knowledge-Base-Python.ipynb) pour la mémoire du débat, et [4-Dialogues-Protocolises](./Argumentation-04-Dialogues-Protocolises-Python.ipynb) pour les règles de l'échange.
- Le tronc (`2025-Epita-Intelligence-Symbolique`) pour les canaux hiérarchique, collaboration, données et pub/sub dont ce carnet distille le contrat commun.

**Référence** — FIPA (2002). « FIPA ACL Message Structure Specification », document SC00061G, Foundation for Intelligent Physical Agents — la structure de message dont la section 1 mesure la version minimale.
